# Web Scraping — CNIL-aligned Privacy Risk Scorer

Built on top of the proven, working scraper (1152/1400 sites successfully
scraped earlier). Adds CNIL-specific compliance checks on top of the
existing tracker/consent/policy detection:

- Differential test: does clicking "reject" actually stop trackers?
- Pre-ticked checkboxes (explicitly prohibited by CNIL)
- Reject button as visible/prominent as accept button (CNIL requirement)
- Cookies set BEFORE any consent is given
- TCF API presence (IAB Transparency & Consent Framework)
- robots.txt presence (basic technical hygiene check)

## Step 0 — Imports

In [36]:
import json
import time
import re
from urllib.parse import urlparse, urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import WebDriverException
from bs4 import BeautifulSoup

import pandas as pd
import requests

## Step 1 — Driver creation (unchanged, proven working)

In [37]:
def create_driver():
    """Creates a headless Chrome browser instance with network logging enabled."""
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    options.set_capability('goog:loggingPrefs', {'performance': 'ALL'})
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(15)
    return driver


def is_driver_alive(driver):
    try:
        driver.title
        return True
    except Exception:
        return False


def ensure_driver(driver):
    if driver is None or not is_driver_alive(driver):
        try:
            driver.quit()
        except Exception:
            pass
        return create_driver()
    return driver

## Step 2 — Tracker detection setup (unchanged, proven working)

In [38]:
with open('raw_data/disconnect_services.json', 'r') as f:
    disconnect_data = json.load(f)


def build_domain_lookup(disconnect_data):
    domain_to_category = {}
    for category, services in disconnect_data['categories'].items():
        for service in services:
            for company_name, urls_dict in service.items():
                for homepage, domains in urls_dict.items():
                    for domain in domains:
                        domain_to_category[domain] = category
    return domain_to_category


domain_to_category = build_domain_lookup(disconnect_data)
print(f'Loaded {len(domain_to_category)} known tracker domains')


def is_known_ad_network(domain):
    for known_domain, category in domain_to_category.items():
        if known_domain in domain and category in ('Advertising', 'Analytics'):
            return True
    return False


def extract_domains_from_logs(driver):
    """Reads Chrome's performance log SINCE THE LAST CALL (buffer clears each time —
    this is what makes a before/after differential comparison possible)."""
    try:
        logs = driver.get_log('performance')
    except Exception:
        return set()
    
    domains = set()
    for entry in logs:
        try:
            message = json.loads(entry['message'])['message']
            if message.get('method') == 'Network.requestWillBeSent':
                url = message['params']['request']['url']
                domain = urlparse(url).netloc
                if domain:
                    domains.add(domain)
        except (KeyError, json.JSONDecodeError):
            continue
    return domains


def count_trackers_in_domains(domains, main_domain):
    """Given a set of domains contacted, returns (third_party_count, tracker_count)."""
    third_party = [d for d in domains if main_domain not in d]
    trackers = [d for d in third_party if is_known_ad_network(d)]
    return len(third_party), len(trackers)

Loaded 4443 known tracker domains


## Step 3 — Cookie consent keyword detection (unchanged, proven working)

In [ ]:
COOKIE_BANNER_KEYWORDS = [
    'cookie', 'cookies', 'consent', 'gdpr', 'rgpd',
    'confidentialite', 'datenschutz', 'privacy'
]
REJECT_BUTTON_KEYWORDS = [
    'reject', 'refuser', 'refuse', 'decline', 'disagree',
    'tout refuser', 'alle ablehnen', 'ablehnen', 'continuer sans accepter',
    'rechazar', 'rifiuta', 'weigeren', 'rejeitar',
    "don't accept", 'do not accept', 'do not agree', "i don't agree","N’accepter que les cookies essentiels",
    "Nur erforderliche"

]

ACCEPT_BUTTON_KEYWORDS = [
    'accept', 'accepter', 'agree', 'allow all', 'i agree',
    'tout accepter', 'alle akzeptieren', 'aceptar', 'accetta', 'accepteren', 'aceitar',
    'ok, got it', 'got it'
]

KNOWN_CMP_NAMES = [
    'didomi', 'onetrust', 'cookiebot', 'usercentrics',
    'trustarc', 'klaro', 'osano', 'cookieyes',
    'complianz', 'iubenda', 'quantcast',
    'funding choices', 'fundingchoices',
    'setupad', 'conversant', 'cookie notice',
    'axeptio', 'sourcepoint', 'commanders act', 'trustcommander'
]

## Step 4 — NEW: simple clickable-element finder

Deliberately simple — main document + first-level iframes only, plain
Selenium text matching. No shadow DOM / JS recursion (that approach caused
instability and false negatives earlier). This will miss shadow-DOM-only
banners (a known, documented limitation) but reliably handles the vast
majority of sites, consistent with the rest of this proven scraper.

In [40]:
def safe_default_content(driver):
    """Switches to the main document, safely -- never raises even if the
    driver session is in a bad/slow state. Use this everywhere instead of
    a bare driver.switch_to.default_content(), since that single call has
    caused several TimeoutExceptions tonight when the browser is degraded."""
    try:
        driver.switch_to.default_content()
    except WebDriverException:
        pass


def find_clickable_button(driver, keywords):
    """
    Searches for a clickable element whose text matches one of `keywords`,
    in the main document, then in each first-level iframe.
    Returns (element, size_dict) or (None, None).
    Leaves the driver switched into the frame where the element was found —
    call driver.switch_to.default_content() once done with it.

    Every driver call is wrapped in try/except -- ANY of them can time out
    if the browser session is in a slow/degraded state, not just driver.get().
    """
    safe_default_content(driver)

    try:
        n_frames = len(driver.find_elements('tag name', 'iframe'))
    except WebDriverException:
        n_frames = 0

    contexts = [None] + list(range(n_frames))
    for ctx in contexts:
        safe_default_content(driver)

        if ctx is not None:
            try:
                driver.switch_to.frame(ctx)
            except WebDriverException:
                continue
        try:
            clickable = driver.find_elements(
                'css selector',
                "button, a[role='button'], [role='button'], input[type='button']"
            )
            for el in clickable:
                try:
                    txt = el.text.strip().lower()
                    if txt and any(k in txt for k in keywords):
                        return el, el.size
                except Exception:
                    continue
        except WebDriverException:
            continue

    safe_default_content(driver)
    return None, None


def click_element_safely(driver, element):
    try:
        driver.execute_script('arguments[0].click();', element)
        return True
    except Exception:
        try:
            element.click()
            return True
        except Exception:
            return False


## Step 5 — NEW: cookie classification

CNIL rule: essential/technical cookies may be set before consent; anything
else (analytics, advertising) should NOT be set until the user consents.
Long-lived cookies (>13 months) also violate CNIL's own recommended limit.

In [43]:
ESSENTIAL_NAME_HINTS = {'cf_', 'session', 'phpsessid', 'csrf', '__cf', 'consent'}


def classify_cookies(cookies, main_domain):
    """
    Returns (total, non_essential_count, long_lived_count).

    A cookie is treated as 'likely non-essential' if EITHER:
    - its domain matches a known tracker from the Disconnect.me list, OR
    - it's a genuine third-party cookie (domain isn't the site itself), OR
    - it's first-party but its name doesn't look essential either

    This is a heuristic grounded in the verified Disconnect.me tracker list
    (domain reputation) rather than guessing purely from cookie names.
    True purpose classification isn't observable from outside the site.
    """
    non_essential = 0
    long_lived = 0

    for c in cookies:
        cookie_domain = c.get('domain', '').lstrip('.')
        name_lower = c.get('name', '').lower()

        is_first_party = main_domain in cookie_domain
        is_known_tracker_domain = is_known_ad_network(cookie_domain)
        looks_essential_by_name = any(hint in name_lower for hint in ESSENTIAL_NAME_HINTS)

        if is_known_tracker_domain or (not is_first_party) or (is_first_party and not looks_essential_by_name):
            non_essential += 1

        expiry = c.get('expiry')
        if expiry and (expiry - time.time()) / 86400 > 13 * 30:  # 13 months, CNIL's own limit
            long_lived += 1

    return len(cookies), non_essential, long_lived

## Step 6 — Privacy policy link (unchanged, proven working)

In [44]:
POLICY_LINK_WORDS = [
    'privacy policy', 'privacy notice',
    'politique de confidentialite', 'politique de confidentialité',
    'mentions legales', 'mentions légales',
    'datenschutz', 'datenschutzerklarung', 'datenschutzerklärung',
    'politica de privacidad', 'política de privacidad', 'aviso de privacidad',
    'informativa sulla privacy', 'politica sulla privacy',
    'privacybeleid', 'privacyverklaring',
    'privacy', 'cookies'
]


def find_policy_link(driver, page_url):
    """Looks for a privacy policy link on the current page. Returns the URL or None."""
    try:
        html = driver.page_source
    except Exception:
        return None
    
    soup = BeautifulSoup(html, 'html.parser')
    all_links = soup.find_all('a', href=True)
    
    matches = {}
    for link in all_links:
        link_text = link.get_text().lower()
        for word in POLICY_LINK_WORDS:
            if word in link_text:
                full_url = urljoin(page_url, link['href'])
                matches[word] = full_url
    
    for word in POLICY_LINK_WORDS:
        if word in matches:
            return matches[word]
    return None

## Step 7 — The main CNIL-aligned audit function

This is the new core function, replacing `collect_all`. Combines the
proven detection logic with the CNIL-specific checks.

In [45]:
def audit_site(driver, url):
    """
    Full CNIL-aligned audit of one URL. Returns a dict (one row of data).

    Everything after the initial page load is wrapped in one outer
    try/except. Individual sub-steps already have their own protection
    where practical, but ANY driver call can time out unpredictably when
    the browser session is in a slow/degraded state -- rather than keep
    patching each new failure point one at a time, this outer safety net
    guarantees one bad site can never crash the whole 1500-URL loop again.
    Partial results collected before the failure are still kept.
    """
    row = {'url': url, 'reachable': False, 'error': None}
    main_domain = urlparse(url).netloc.replace('www.', '')

    try:
        driver.get(url)
        time.sleep(5)
    except Exception as e:
        row['error'] = str(e)
        return row
    row['reachable'] = True

    try:
        # --- cookies + trackers BEFORE any interaction ---
        cookies_before = driver.get_cookies()
        total_before, non_essential_before, long_lived_before = classify_cookies(cookies_before, main_domain)
        row['cookies_before_interaction'] = total_before
        row['non_essential_cookies_before_interaction'] = non_essential_before
        row['long_lived_cookies_count'] = long_lived_before

        domains_before = extract_domains_from_logs(driver)
        third_party_before, trackers_before = count_trackers_in_domains(domains_before, main_domain)
        row['tracker_count'] = third_party_before
        row['ad_network_count'] = trackers_before

        # --- CMP name detection (text-based, proven working) ---
        html_lower = driver.page_source.lower()
        detected_cmp = None
        for cmp_name in KNOWN_CMP_NAMES:
            if cmp_name in html_lower:
                detected_cmp = cmp_name
                break
        row['cmp_detected'] = detected_cmp

        # --- TCF API presence ---
        try:
            row['tcf_api_present'] = bool(
                driver.execute_script("return typeof window.__tcfapi === 'function';")
            )
        except WebDriverException:
            row['tcf_api_present'] = False

        # --- find accept / reject buttons ---
        accept_el, accept_size = find_clickable_button(driver, ACCEPT_BUTTON_KEYWORDS)
        safe_default_content(driver)
        reject_el, reject_size = find_clickable_button(driver, REJECT_BUTTON_KEYWORDS)
        safe_default_content(driver)

        row['has_accept_button'] = accept_el is not None
        row['has_reject_button'] = reject_el is not None
        row['has_cookie_banner'] = row['has_accept_button'] or row['has_reject_button']

        # --- CNIL requirement: is reject as visually prominent as accept? ---
        row['reject_as_visible_as_accept'] = None
        if accept_size and reject_size:
            a_area = accept_size.get('width', 0) * accept_size.get('height', 0)
            r_area = reject_size.get('width', 0) * reject_size.get('height', 0)
            if max(a_area, r_area) > 0:
                row['reject_as_visible_as_accept'] = (min(a_area, r_area) / max(a_area, r_area)) >= 0.7

        # --- CNIL requirement: no pre-ticked boxes ---
        try:
            safe_default_content(driver)
            checkboxes = driver.find_elements('css selector', "input[type='checkbox']")
            row['prechecked_boxes_detected'] = any(
                cb.is_selected() for cb in checkboxes if cb.is_displayed()
            )
        except WebDriverException:
            row['prechecked_boxes_detected'] = None
        safe_default_content(driver)

        # --- THE differential test: does clicking reject actually work? ---
        row['reject_click_attempted'] = False
        row['reject_click_succeeded'] = False
        row['tracker_count_after_reject'] = None
        row['non_essential_cookies_after_reject'] = None

        if reject_el is not None:
            row['reject_click_attempted'] = True
            reject_el_fresh, _ = find_clickable_button(driver, REJECT_BUTTON_KEYWORDS)
            if reject_el_fresh is not None:
                row['reject_click_succeeded'] = click_element_safely(driver, reject_el_fresh)
            safe_default_content(driver)
            time.sleep(3)

            cookies_after = driver.get_cookies()
            _, non_essential_after, _ = classify_cookies(cookies_after, main_domain)
            row['non_essential_cookies_after_reject'] = non_essential_after

            domains_after = extract_domains_from_logs(driver)
            _, trackers_after = count_trackers_in_domains(domains_after, main_domain)
            row['tracker_count_after_reject'] = trackers_after

        # --- privacy policy link ---
        safe_default_content(driver)
        policy_link = find_policy_link(driver, url)
        row['has_privacy_policy'] = policy_link is not None

        # --- robots.txt (cheap technical hygiene signal) ---
        try:
            driver.get(f"{urlparse(url).scheme}://{main_domain}/robots.txt")
            time.sleep(1)
            row['robots_txt_present'] = 'user-agent' in driver.page_source.lower()
        except WebDriverException:
            row['robots_txt_present'] = None

    except Exception as e:
        # Outer safety net -- whatever partial data was collected above is
        # kept in `row`; we just record that something failed partway through
        row['error'] = f'partial failure: {str(e)[:200]}'

    return row

## Step 8 — Test on a few known sites

In [47]:
driver = create_driver()

test_sites = ['https://www.lemonde.fr/', 'https://www.abeille-assurances.fr/', 'https://www.sephora.fr/']

test_results = []
for site in test_sites:
    driver = ensure_driver(driver)
    print('Auditing:', site)
    result = audit_site(driver, site)
    test_results.append(result)

driver.quit()

df_test = pd.DataFrame(test_results)
df_test

Auditing: https://www.lemonde.fr/
Auditing: https://www.abeille-assurances.fr/
Auditing: https://www.sephora.fr/


,url,reachable,error,cookies_before_interaction,non_essential_cookies_before_interaction,long_lived_cookies_count,tracker_count,ad_network_count,cmp_detected,tcf_api_present,...,has_reject_button,has_cookie_banner,reject_as_visible_as_accept,prechecked_boxes_detected,reject_click_attempted,reject_click_succeeded,tracker_count_after_reject,non_essential_cookies_after_reject,has_privacy_policy,robots_txt_present
0,https://www.lemonde.fr/,True,None,8,8,6,4,4,None,True,...,False,True,None,False,False,False,NaN,NaN,True,True
1,https://www.abeille-assurances.fr/,True,None,13,13,3,9,3,didomi,True,...,True,True,True,False,True,True,0.0,14.0,True,None
2,https://www.sephora.fr/,True,None,19,19,1,10,6,trustcommander,False,...,True,True,False,False,True,True,2.0,28.0,True,True


In [48]:
driver = create_driver()

test_gap_results = []
for url in gap_urls[:10]:
    driver = ensure_driver(driver)
    print('Testing:', url)
    row = audit_gap_site(driver, url)
    test_gap_results.append(row)
    print(' ->', row)

driver.quit()

df_test_gap = pd.DataFrame(test_gap_results)
print()
print('Found a clickable reject button on:', df_test_gap['reject_click_attempted_v2'].sum(), '/ 10')

Testing: https://europa.eu
 -> {'url': 'https://europa.eu', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://telekom.de
 -> {'url': 'https://telekom.de', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://t-online.de
 -> {'url': 'https://t-online.de', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://bbc.co.uk
 -> {'url': 'https://bbc.co.uk', 'reject_click_attempted_v2': True, 'reject_click_succeeded_v2': True, 'tracker_count_after_reject_v2': 16, 'gap_error': None}
Testing: https://www.gov.uk
 -> {'url': 'https://www.gov.uk', 'reject_click_attempted_v2': True, 'reject_click_succeeded_v2': True, 'tracker_count_after_reject_v2': 0, 'gap_error': None}
Testing: https://amazon.co.uk
 -> {'url': 'https://

## Step 9 — Full run on all URLs
Same pattern as before: progress saved every 100 sites.

In [9]:
df_urls = pd.read_csv('clean_data/urls.csv')
url_list = df_urls['url'].tolist()
print(f'Total URLs to scrape: {len(url_list)}')

Total URLs to scrape: 1500


driver = create_driver()

all_results = []

for i, url in enumerate(url_list):
    driver = ensure_driver(driver)
    row = audit_site(driver, url)
    all_results.append(row)

    if (i + 1) % 100 == 0:
        print(f'Done {i+1} / {len(url_list)}')
        pd.DataFrame(all_results).to_csv('raw_data/cnil_audit_progress.csv', index=False)

driver.quit()

df_final = pd.DataFrame(all_results)
df_final.to_csv('raw_data/cnil_audit_results.csv', index=False)
print('Saved cnil_audit_results.csv —', df_final.shape)


df_final = pd.DataFrame(all_results)
df_final.to_csv('raw_data/cnil_audit_progress.csv', index=False)
print('Saved cnil_audit_results.csv —', df_final.shape)

In [10]:
df_progress = pd.read_csv('raw_data/cnil_audit_progress.csv')
already_done = len(df_progress)
print(f'Already scraped: {already_done} rows')

Already scraped: 502 rows


In [11]:
driver = create_driver()

all_results = list(df_progress.to_dict('records'))  # start with what you already have

remaining_urls = url_list[already_done:]
print(f'Remaining to scrape: {len(remaining_urls)}')

for i, url in enumerate(remaining_urls):
    driver = ensure_driver(driver)
    row = audit_site(driver, url)
    all_results.append(row)

    if (i + 1) % 100 == 0:
        print(f'Done {i+1} / {len(remaining_urls)} (total so far: {len(all_results)})')
        pd.DataFrame(all_results).to_csv('raw_data/cnil_audit_progress.csv', index=False)

driver.quit()

df_final = pd.DataFrame(all_results)
df_final.to_csv('raw_data/cnil_audit_results.csv', index=False)
print('Saved cnil_audit_results.csv —', df_final.shape)

Remaining to scrape: 998
Done 100 / 998 (total so far: 602)
Done 200 / 998 (total so far: 702)
Done 300 / 998 (total so far: 802)
Done 400 / 998 (total so far: 902)
Done 500 / 998 (total so far: 1002)
Done 600 / 998 (total so far: 1102)
Done 700 / 998 (total so far: 1202)
Done 800 / 998 (total so far: 1302)
Done 900 / 998 (total so far: 1402)
Saved cnil_audit_results.csv — (1500, 21)


In [13]:
df_final = pd.DataFrame(all_results)
df_final.to_csv('raw_data/cnil_audit_results.csv', index=False)
print('Saved cnil_audit_results.csv —', df_final.shape)

Saved cnil_audit_results.csv — (1500, 21)


In [15]:
df = pd.read_csv('raw_data/cnil_audit_results.csv')

print('Total rows:', len(df))
print('Reachable:', df['reachable'].sum())
print('Has an error:', df['error'].notna().sum())
print('Has cookie banner detected:', df['has_cookie_banner'].sum())
print('Reject click attempted:', df['reject_click_attempted'].sum())
print('Reject click succeeded:', df['reject_click_succeeded'].sum())

Total rows: 1500
Reachable: 1238
Has an error: 263
Has cookie banner detected: 392
Reject click attempted: 263
Reject click succeeded: 258


In [16]:
df_old = pd.read_csv('clean_data/scraped_results_clean.csv')
print('Old scraper banner rate:', df_old['has_cookie_banner'].mean() * 100, '%')
print('New scraper banner rate:', (df['has_cookie_banner'].sum() / df['reachable'].sum()) * 100, '%')

Old scraper banner rate: 83.33333333333334 %
New scraper banner rate: 31.663974151857836 %


## Step 1 — Merge the two existing files

In [27]:
import pandas as pd
from urllib.parse import urlparse

df_old = pd.read_csv('clean_data/scraped_results_clean.csv')
df_new = pd.read_csv('raw_data/cnil_audit_results.csv')

# normalize both to bare domain so they match regardless of URL formatting
df_old['domain'] = df_old['url'].apply(lambda u: urlparse(u).netloc.replace('www.', ''))
df_new['domain'] = df_new['url'].apply(lambda u: urlparse(u).netloc.replace('www.', ''))

old_cols = ['domain', 'has_cookie_banner', 'has_reject_button', 'has_accept_button',
            'cmp_detected', 'tracker_count', 'ad_network_count']
new_cols = ['domain', 'url', 'reachable', 'reject_click_attempted', 'reject_click_succeeded',
            'tracker_count_after_reject', 'non_essential_cookies_before_interaction',
            'non_essential_cookies_after_reject', 'prechecked_boxes_detected',
            'reject_as_visible_as_accept', 'tcf_api_present', 'robots_txt_present',
            'has_privacy_policy']

df_merged = df_new[new_cols].merge(
    df_old[old_cols], on='domain', how='left', suffixes=('_new', '_old')
)

print(df_merged.shape)
df_merged.head()

(1500, 19)


,domain,url,reachable,reject_click_attempted,reject_click_succeeded,tracker_count_after_reject,non_essential_cookies_before_interaction,non_essential_cookies_after_reject,prechecked_boxes_detected,reject_as_visible_as_accept,tcf_api_present,robots_txt_present,has_privacy_policy,has_cookie_banner,has_reject_button,has_accept_button,cmp_detected,tracker_count,ad_network_count
0,youtu.be,https://youtu.be,True,True,True,2.0,7.0,7.0,False,True,False,True,False,True,True,True,none,9.0,3.0
1,europa.eu,https://europa.eu,True,False,False,NaN,3.0,NaN,False,NaN,False,True,True,True,False,True,none,10.0,2.0
2,telekom.de,https://telekom.de,True,False,False,NaN,5.0,NaN,False,NaN,False,True,True,True,False,True,none,9.0,4.0
3,iiko.it,https://iiko.it,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,nominetdns.uk,https://nominetdns.uk,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 — Identify the gap
Sites where the OLD scraper found a banner, but the NEW scraper's
click-search found nothing to click (no reject attempt possible).

In [28]:
print(df_merged.columns.tolist())

['domain', 'url', 'reachable', 'reject_click_attempted', 'reject_click_succeeded', 'tracker_count_after_reject', 'non_essential_cookies_before_interaction', 'non_essential_cookies_after_reject', 'prechecked_boxes_detected', 'reject_as_visible_as_accept', 'tcf_api_present', 'robots_txt_present', 'has_privacy_policy', 'has_cookie_banner', 'has_reject_button', 'has_accept_button', 'cmp_detected', 'tracker_count', 'ad_network_count']


In [19]:
gap_mask = (df_merged['has_cookie_banner'] == True) & (df_merged['reject_click_attempted'] == False)
df_gap = df_merged[gap_mask]

gap_urls = df_gap['url'].tolist()
print(f'Sites to re-scan with shadow-DOM search: {len(gap_urls)}')

Sites to re-scan with shadow-DOM search: 710


## Step 3 — Shadow-DOM-aware clickable finder
Same JS-piercing technique from earlier tonight (with the document.innerHTML
bug already fixed), used here as an UPGRADE — tries the simple method first
(fast), falls back to the shadow-DOM search only if that fails.

In [29]:
import time
from selenium.common.exceptions import WebDriverException


def find_buttons_deep(driver):
    """
    Shadow-DOM-piercing search in the CURRENT frame/context.
    Returns a list of WebElements. Fixed version: starts from
    document.documentElement (document itself has no .innerHTML), and the
    selector now ALSO catches input[type="submit"] -- Amazon and many other
    sites use <input type="submit" value="Refuser"> for their reject button,
    not a <button> tag, which the earlier version completely missed.
    """
    deep_search_js = """
    function deepQueryButtons(root) {
        let results = [];
        function walk(node) {
            if (!node) return;
            const matches = node.querySelectorAll(
                'button, a[role="button"], [role="button"], ' +
                'input[type="button"], input[type="submit"]'
            );
            for (const el of matches) { results.push(el); }
            const all = node.querySelectorAll('*');
            for (const el of all) {
                if (el.shadowRoot) { walk(el.shadowRoot); }
            }
        }
        walk(root);
        return results;
    }
    return deepQueryButtons(document.documentElement);
    """
    try:
        return driver.execute_script(deep_search_js)
    except WebDriverException:
        return []


def get_element_label(el):
    """
    Returns the best available text label for an element.
    <button> elements have real .text -- but <input type="submit"> elements
    do NOT (inputs have no text nodes); their visible label lives in the
    'value' attribute instead. aria-label is checked too, as a further
    fallback some sites use for accessibility.
    """
    try:
        txt = el.text.strip()
        if txt:
            return txt.lower()
    except Exception:
        pass
    try:
        value = el.get_attribute('value')
        if value:
            return value.strip().lower()
    except Exception:
        pass
    try:
        aria = el.get_attribute('aria-label')
        if aria:
            return aria.strip().lower()
    except Exception:
        pass
    return ''


def find_clickable_button_deep(driver, keywords, max_iframe_depth=2):
    """
    Full search: main doc + shadow roots (any depth) + iframes
    (nested up to max_iframe_depth, each with their own shadow roots too).
    """

    def search_here():
        for el in find_buttons_deep(driver):
            txt = get_element_label(el)
            if txt and any(k in txt for k in keywords):
                return el, el.size
        return None, None

    def recurse(depth):
        found, size = search_here()
        if found is not None:
            return found, size
        if depth >= max_iframe_depth:
            return None, None
        try:
            iframes = driver.find_elements('tag name', 'iframe')
        except WebDriverException:
            return None, None
        for i in range(len(iframes)):
            try:
                current_iframes = driver.find_elements('tag name', 'iframe')
                driver.switch_to.frame(current_iframes[i])
            except (WebDriverException, IndexError):
                continue
            result, size = recurse(depth + 1)
            if result is not None:
                return result, size
            try:
                driver.switch_to.parent_frame()
            except WebDriverException:
                pass
        return None, None

    try:
        driver.switch_to.default_content()
    except WebDriverException:
        return None, None
    result, size = recurse(depth=0)
    try:
        if result is None:
            driver.switch_to.default_content()
    except WebDriverException:
        pass
    return result, size


## Step 4 — Re-run the differential test on ONLY the gap sites

In [30]:
def audit_gap_site(driver, url, main_domain_hint=None):
    """
    Focused re-audit: finds a reject button via the deep shadow-DOM search,
    clicks it, re-measures trackers. Same differential logic as audit_site,
    but using the deeper button finder this time.
    """
    row = {'url': url}
    main_domain = main_domain_hint or urlparse(url).netloc.replace('www.', '')

    try:
        driver.get(url)
        time.sleep(5)
    except Exception as e:
        row['gap_error'] = str(e)
        return row

    try:
        reject_el, reject_size = find_clickable_button_deep(driver, REJECT_BUTTON_KEYWORDS)
        row['reject_click_attempted_v2'] = reject_el is not None

        if reject_el is not None:
            domains_before = extract_domains_from_logs(driver)
            _, trackers_before = count_trackers_in_domains(domains_before, main_domain)

            clicked = click_element_safely(driver, reject_el)
            row['reject_click_succeeded_v2'] = clicked
            safe_default_content(driver)
            time.sleep(3)

            domains_after = extract_domains_from_logs(driver)
            _, trackers_after = count_trackers_in_domains(domains_after, main_domain)
            row['tracker_count_after_reject_v2'] = trackers_after
        else:
            row['reject_click_succeeded_v2'] = False
            row['tracker_count_after_reject_v2'] = None

        row['gap_error'] = None
    except Exception as e:
        row['gap_error'] = f'partial failure: {str(e)[:200]}'

    return row

In [31]:
print(create_driver)
print(ensure_driver)
print(extract_domains_from_logs)
print(count_trackers_in_domains)
print(click_element_safely)
print(safe_default_content)
print(REJECT_BUTTON_KEYWORDS)

<function create_driver at 0x15fb772e0>
<function ensure_driver at 0x15fb774c0>
<function extract_domains_from_logs at 0x15fb77740>
<function count_trackers_in_domains at 0x15fb777e0>
<function click_element_safely at 0x15fb77a60>
<function safe_default_content at 0x15fb77920>
['reject all', 'refuser', 'refuse all', 'decline', 'tout refuser', 'alle ablehnen', 'ablehnen', 'continuer sans accepter', 'rechazar', 'rifiuta', 'weigeren', 'rejeitar']


In [33]:
driver = create_driver()

test_gap_results = []
for url in gap_urls[:10]:
    driver = ensure_driver(driver)
    print('Testing:', url)
    row = audit_gap_site(driver, url)
    test_gap_results.append(row)
    print(' ->', row)

driver.quit()

df_test_gap = pd.DataFrame(test_gap_results)
print()
print('Found a clickable reject button on:', df_test_gap['reject_click_attempted_v2'].sum(), '/ 10')

Testing: https://europa.eu
 -> {'url': 'https://europa.eu', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://telekom.de
 -> {'url': 'https://telekom.de', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://t-online.de
 -> {'url': 'https://t-online.de', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://bbc.co.uk
 -> {'url': 'https://bbc.co.uk', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://www.gov.uk
 -> {'url': 'https://www.gov.uk', 'reject_click_attempted_v2': False, 'reject_click_succeeded_v2': False, 'tracker_count_after_reject_v2': None, 'gap_error': None}
Testing: https://amazon.co.uk
 -> {'url': 

In [26]:
print(REJECT_BUTTON_KEYWORDS)

['reject all', 'refuser', 'refuse all', 'decline', 'tout refuser', 'alle ablehnen', 'ablehnen', 'continuer sans accepter', 'rechazar', 'rifiuta', 'weigeren', 'rejeitar']


In [ ]:
driver = create_driver()

gap_results = []
for i, url in enumerate(gap_urls):
    driver = ensure_driver(driver)
    row = audit_gap_site(driver, url)
    gap_results.append(row)

    if (i + 1) % 50 == 0:
        print(f'Done {i+1} / {len(gap_urls)}')
        pd.DataFrame(gap_results).to_csv('raw_data/gap_reaudit_progress.csv', index=False)

driver.quit()

df_gap_results = pd.DataFrame(gap_results)
df_gap_results.to_csv('raw_data/gap_reaudit_results.csv', index=False)
print('Recovered via shadow-DOM search:', df_gap_results['reject_click_attempted_v2'].sum(), '/', len(df_gap_results))

## Step 5 — Final merge: combine everything into one complete file

In [ ]:
df_complete = df_merged.merge(df_gap_results, on='url', how='left')

# Where the gap re-scan found something new, use it. Otherwise keep the original.
df_complete['reject_click_attempted_final'] = df_complete['reject_click_attempted_v2'].fillna(
    df_complete['reject_click_attempted']
)
df_complete['reject_click_succeeded_final'] = df_complete['reject_click_succeeded_v2'].fillna(
    df_complete['reject_click_succeeded']
)
df_complete['tracker_count_after_reject_final'] = df_complete['tracker_count_after_reject_v2'].fillna(
    df_complete['tracker_count_after_reject']
)

print('Original differential test coverage:', df_complete['reject_click_attempted'].sum())
print('Final differential test coverage:', df_complete['reject_click_attempted_final'].sum())

df_complete.to_csv('clean_data/cnil_audit_complete.csv', index=False)
print('Saved cnil_audit_complete.csv —', df_complete.shape)